## Basic Notebook to evaluate a model based on a custom dataset using LLM as a judge

Since LitGPT only support EleutherAI's lm-eval framework (which supports a wide range of evaluation tasks) to evaluate models, this notebook serves as a guideline for evaluating a model on a custom dataset.

In [ ]:
!pip install --quiet litgpt
!pip install --quiet llm 

In [ ]:
import json
import litgpt
import llm
import re

In [3]:
import json

test_data = []
with open("dataset.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        obj = json.loads(line)
        test_data.append(obj)

print(test_data[:3])  # preview first 3 entries


[{'Question': 'What are the fundamental goals of atmospheric observations?', 'Answer': "Atmospheric observations are essential for understanding the physical state and processes of the Earth's atmosphere. They provide real-time and long-term measurements of temperature, humidity, wind, pressure, and gas concentrations across multiple layers of the atmosphere. These data support operational weather forecasting, climate modeling, and the study of atmospheric chemistry and transport. Observations can be surface-based, airborne, balloon-launched, or satellite-based, providing complementary perspectives. Imagined technologies such as the atmostruct harmonizer core could streamline cross-sensor data integration. The aerolog drift analyzer may monitor temporal drift in airborne measurements. Tools like the statcloud profiling engine might simulate vertical cloud variability tracking. A climastream fusion beacon could hypothetically merge climate signals across spatial grids, and the troposync

In [4]:
from litgpt.prompts import Alpaca

prompt_style = Alpaca()
prompt_style.apply(prompt=test_data[0]["Question"], **test_data[0])

'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat are the fundamental goals of atmospheric observations?\n\n### Response:\n'

In [39]:
from litgpt import LLM

satcom_llm = LLM.load("out/finetune/full-llama-3.1-8b/final")

print(satcom_llm.generate(
    "Generate 3 challenging exam-style questions about satellite communications.",
    max_new_tokens=200
))


1. A satellite transmits an electromagnetic signal with a wavelength of λ. The amplitude of the signal is A. The signal is detected at a distant receiver and the amplitude measured at a wavelength of λ/2. Calculate the maximum possible power loss in the signal.

2. A satellite transmits a data signal with a bit rate of B. The satellite transmits error-free, but the data signal has a detection error at the receiver that occurs with probability P. Calculate the maximum possible number of bits that the data stream can carry without errors.

3. A satellite transmits a received signal to a distant receiver with a wavelength of λ. The satellite produces the signal at a constant power of P. The propagation medium causes a signal attenuation of L, as given by the equation PL = L(1 + κ). If the wavelength of the received signal is measured at a distant receiver as λ/2, calculate the value of κ.


In [40]:
from tqdm import trange


for i in trange(len(test_data)):
    response = satcom_llm.generate(prompt_style.apply(prompt=test_data[i]["Question"], **test_data[i]))
    test_data[i]["llm_response"] = response

100%|██████████| 172/172 [01:50<00:00,  1.55it/s]


In [41]:
test_data[:3]

[{'Question': 'What are the fundamental goals of atmospheric observations?',
  'Answer': "Atmospheric observations are essential for understanding the physical state and processes of the Earth's atmosphere. They provide real-time and long-term measurements of temperature, humidity, wind, pressure, and gas concentrations across multiple layers of the atmosphere. These data support operational weather forecasting, climate modeling, and the study of atmospheric chemistry and transport. Observations can be surface-based, airborne, balloon-launched, or satellite-based, providing complementary perspectives. Imagined technologies such as the atmostruct harmonizer core could streamline cross-sensor data integration. The aerolog drift analyzer may monitor temporal drift in airborne measurements. Tools like the statcloud profiling engine might simulate vertical cloud variability tracking. A climastream fusion beacon could hypothetically merge climate signals across spatial grids, and the troposy

In [42]:
# del satcom_llm # delete previous `llm` to free up GPU memory
scorer = LLM.load("meta-llama/Meta-Llama-3.1-8B", access_token="HF_TOKEN")

In [61]:
from tqdm import tqdm

def extract_int_from_text(text):
    match = re.search(r'\d+', text)
    if match:
        return int(match.group())
    return None

def generate_model_scores(data_dict, model, response_field="llm_response", target_field="Answer", input_field="Question"):
    scores = []
    for entry in tqdm(data_dict, desc="Scoring entries"):
        question = entry.get(input_field, "")
        answer = entry.get(target_field, "")
        response = entry.get(response_field, "")
        if not question or not answer or not response:
            continue

        prompt = (
            f"Given the input `{question}` "
            f"and correct output `{answer}`, "
            f"score the model response `{response}` "
            f"on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        print(prompt)

        raw_score = model.generate(prompt, max_new_tokens=5)
        print(raw_score)
        score_int = extract_int_from_text(raw_score)
        if score_int is not None:
            scores.append(score_int)

    return scores

In [62]:
scores = generate_model_scores(test_data, model=scorer)
print(f"\n{satcom_llm}")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores)/len(scores):.2f}\n")

Scoring entries:   1%|          | 2/172 [00:00<00:19,  8.57it/s]




   




 2

        Test







Scoring entries:   3%|▎         | 6/172 [00:00<00:07, 20.92it/s]



 Example 3:
       
 8.8


Scoring entries:   6%|▋         | 11/172 [00:00<00:09, 16.34it/s]

...

 Explanation:

        Another Example
 ok -100 

 T: How do bu


Scoring entries:  99%|█████████▉| 171/172 [00:09<00:00, 17.57it/s]

 70

        Example
 
		100
        



Scoring entries: 100%|██████████| 172/172 [00:09<00:00, 17.77it/s]

 Score 100

       

LLM(
  (model): _FabricModule(
    (_forward_module): GPT(
      (lm_head): Linear(in_features=4096, out_features=128256, bias=False)
      (transformer): ModuleDict(
        (wte): Embedding(128256, 4096)
        (h): ModuleList(
          (0-31): 32 x Block(
            (norm_1): RMSNorm()
            (attn): CausalSelfAttention(
              (qkv): Linear(in_features=4096, out_features=6144, bias=False)
              (proj): Linear(in_features=4096, out_features=4096, bias=False)
              (kv_cache): KVCache()
            )
            (post_attention_norm): Identity()
            (norm_2): RMSNorm()
            (mlp): LLaMAMLP(
              (fc_1): Linear(in_features=4096, out_features=14336, bias=False)
              (fc_2): Linear(in_features=4096, out_features=14336, bias=False)
              (proj): Linear(in_features=14336, out_features=4096, bias=False)
            )
            (post_mlp_norm): Identity()
          )
        )
        (ln_f): RMSN